# PapyrusLab E02 — run `prep-w029` (segmento `pherc1667-w029`)

Notebook generato da `scripts/build_e02_notebooks.py` (non modificare a mano). Piano congelato: `docs/plans/2026-09-06-e02-costruire-il-metro.md`.
Ogni controllo di arresto del piano è un'asserzione: se fallisce, il run si ferma e il log dice dove.

- Output persistiti: `/kaggle/working/e02/out` e `/kaggle/working/e02/logs`
- File pesanti (codice, checkpoint, label, input, cache), non persistiti: `/tmp/e02`


In [ ]:
MODE = "prep-w029"
KIND = "prep"            # prep | infer
SEG = "pherc1667-w029"              # nome della label, es. pherc0814-46527
SHORT = "w029"
SEED = None              # None nel prep
SETS = "train"            # insiemi di pixel misurati: 'held,train' oppure 'train' (segmento di verifica sigillato)
WORK = "/kaggle/working/e02"
HEAVY = "/tmp/e02"
SRC_URL = "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc1667/segments/20251212185248-w029_20251212185248662_flatboi/surface-volumes/2.399um-0.22m-78keV-volume-20251217075048.zarr"      # volume 2,4 um sorgente (solo prep)
LABEL_SHAPE = [21, 9500, 7830]
TORCH_EXPECTED = "2.10.0+cpu"
LABEL_TREE_SHA256 = "26cf3c17c202957f6c8c7fe1d9ded9a4d40b335d97091b012b8cd9ed92c99ee2"
LABEL_FILES, LABEL_BYTES = 13959, 1108191
INPUT_TREE_SHA256 = ""     # atteso solo nei run infer
print("MODE", MODE, "SEG", SEG, "SEED", SEED, "SETS", SETS, "torch atteso", TORCH_EXPECTED)


In [ ]:
%%bash
# Passo 1 — radice misurabile, cache e temporanei dirottati, guardia dei 15 GB
set -e
mkdir -p /kaggle/working/e02/out /kaggle/working/e02/logs /tmp/e02/tmp /tmp/e02/cache/pip /tmp/e02/cache/hf /tmp/e02/checkpoints /tmp/e02/labels /tmp/e02/input
cat > /kaggle/working/e02/env.sh <<'EOF'
export WORK=/kaggle/working/e02
export HEAVY=/tmp/e02
export TMPDIR=$HEAVY/tmp PIP_CACHE_DIR=$HEAVY/cache/pip HF_HOME=$HEAVY/cache/hf
export LIMIT_BYTES=$((15*1024*1024*1024))
disk_check () {
  local used
  used=$(( $(du -sb "$WORK" | cut -f1) + $(du -sb "$HEAVY" | cut -f1) ))
  echo "spazio_byte=$used ($1)" | tee -a "$WORK/logs/disk_check.log"
  if [ "$used" -gt "$LIMIT_BYTES" ]; then echo "STOP: superati 15 GB ($1)" | tee -a "$WORK/logs/disk_check.log"; exit 1; fi
}
EOF
source /kaggle/working/e02/env.sh
echo "MODE=prep-w029 SEG=pherc1667-w029 SEED=None start=$(date -u +%FT%TZ)" > $WORK/logs/run_info.txt
disk_check "inizio"
df -h /kaggle/working /tmp | tail -2


In [ ]:
# Passo 1 (segue) — versioni dell'ambiente e rete verso le sorgenti
import sys, platform, json, urllib.request, torch
env = {"python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
       "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0}
print(json.dumps(env, indent=1))
json.dump(env, open(f"{WORK}/logs/env_before_install.json", "w"), indent=1)
assert env["torch"] == TORCH_EXPECTED, f"STOP: PyTorch inatteso {env['torch']} (atteso {TORCH_EXPECTED}): Kaggle ha cambiato immagine, aggiornare il piano"
if KIND == "prep":
    assert not env["cuda_available"], "STOP: il run prep deve girare senza acceleratore"
else:
    assert env["cuda_available"], "STOP: run GPU senza CUDA disponibile"
urls = ["https://huggingface.co/api/models/scrollprize/ink_9um", "https://huggingface.co/api/buckets/scrollprize/datasets"]
if KIND == "prep":
    urls.insert(0, SRC_URL + "/2/.zarray")
for url in urls:
    with urllib.request.urlopen(url, timeout=30) as r:
        print(r.status, url[:90]); assert r.status == 200, f"STOP: rete non raggiunge {url}"


In [ ]:
%%bash
# Passo 2 — checkout parziale di villa al commit congelato
set -e
source /kaggle/working/e02/env.sh
cd $HEAVY
[ -d villa/.git ] || git clone -q --filter=blob:none --no-checkout https://github.com/ScrollPrize/villa.git
cd villa
git sparse-checkout init --cone >/dev/null
git sparse-checkout set ink-detection vesuvius >/dev/null
git checkout -q 3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e
HEAD=$(git rev-parse HEAD); echo "villa HEAD=$HEAD" | tee $WORK/logs/villa_commit.txt
[ "$HEAD" = "3ea17f54a9b3d5fd1aaf73e1d2c8386dbaa9f30e" ] || { echo "STOP: commit villa diverso"; exit 1; }
ls -l ink-detection/koine_machines/inference/infer.py ink-detection/scripts/prepare_9um_isotropic_input.py vesuvius/pyproject.toml ink-detection/uv.lock
sha256sum ink-detection/scripts/prepare_9um_isotropic_input.py | tee $WORK/logs/prepare_script_sha256.txt
disk_check "dopo checkout"


In [ ]:
# Passo 3 (prep) — sole dipendenze del pooling ufficiale: zarr 2.18.7 e numcodecs 0.15.1 (come E00), niente altro
import subprocess, sys, json, re
PY = sys.executable
torch_before = subprocess.run([PY, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
r = subprocess.run([PY, "-m", "pip", "install", "--no-deps", "zarr==2.18.7", "numcodecs==0.15.1"], capture_output=True, text=True)
assert r.returncode == 0, "STOP: pip install zarr/numcodecs fallita\n" + r.stderr[-3000:]
# Tentativo 1 di prep-46527 (7 settembre 2026): zarr 2.18.7 importa `asciitree`, assente sull'immagine CPU di Kaggle.
# Stesso ciclo di E00: ogni modulo mancante si installa con --no-deps nella versione del lock di villa (max 10).
lock = open(f"{HEAVY}/villa/ink-detection/uv.lock", encoding="utf-8").read()
def locked_version(dist):
    m = re.search(r'\[\[package\]\]\nname = "' + re.escape(dist.lower()) + r'"\nversion = "([^"]+)"', lock)
    return m.group(1) if m else None
added = []
for attempt in range(10):
    chk = subprocess.run([PY, "-c", "import zarr, numcodecs, numpy, fsspec, aiohttp; print(zarr.__version__, numcodecs.__version__, numpy.__version__, fsspec.__version__, aiohttp.__version__)"], capture_output=True, text=True)
    if chk.returncode == 0:
        break
    m = re.search(r"No module named '([^'.]+)", chk.stderr)
    assert m, "STOP: import fallito per motivo diverso da modulo mancante\n" + chk.stderr[-2000:]
    mod = m.group(1); assert re.fullmatch(r"[A-Za-z0-9_]+", mod), mod
    dist = next((c for c in (mod, mod.replace("_", "-")) if locked_version(c)), None)
    assert dist, f"STOP: modulo mancante '{mod}' non presente in uv.lock"
    r2 = subprocess.run([PY, "-m", "pip", "install", "--no-deps", f"{dist}=={locked_version(dist)}"], capture_output=True, text=True)
    assert r2.returncode == 0, f"STOP: pip install {dist} fallita\n" + r2.stderr[-2000:]
    added.append({"module": mod, "dist": dist, "version": locked_version(dist)}); print("aggiunto", dist, locked_version(dist))
assert chk.returncode == 0, "STOP: import ancora fallito dopo i tentativi ammessi\n" + chk.stderr[-2000:]
torch_after = subprocess.run([PY, "-c", "import torch; print(torch.__version__)"], capture_output=True, text=True).stdout.strip()
assert torch_after == torch_before == TORCH_EXPECTED, f"STOP: PyTorch cambiato da {torch_before} a {torch_after}"
vers = chk.stdout.strip().split()
assert vers[0] == "2.18.7", f"STOP: zarr {vers[0]} invece di 2.18.7"
info = {"torch_before": torch_before, "torch_after": torch_after, "zarr": vers[0], "numcodecs": vers[1], "numpy": vers[2], "fsspec": vers[3], "aiohttp": vers[4], "added_packages": added}
json.dump(info, open(f"{WORK}/logs/install.json", "w"), indent=1); print(info)


In [ ]:
# Passo 4 (segue) — label del segmento dal dataset Kaggle privato (ricerca ricorsiva: il mount cambia fra sessioni
# CPU e GPU, lezione di E00), verificata file per file contro manifest.json e per contenuto (tree_sha256);
# ripiego: download diretto dal bucket con 4 thread e attesa crescente (HTTP 429), come E00.
import hashlib, os, shutil, json, glob, re, time, urllib.request, concurrent.futures as cf
PREFIX = f"ink_9um/labels/aligned-scrollprizeorg-21slices/{SEG}/"
API = "https://huggingface.co/api/buckets/scrollprize/datasets/tree/" + PREFIX.rstrip("/")
RESOLVE = "https://huggingface.co/buckets/scrollprize/datasets/resolve/"
DEST = f"{HEAVY}/labels/{SEG}"

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

def check_label_tree(dest, source_desc):
    n = sum(len(fs) for _, _, fs in os.walk(dest)); b = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(dest) for f in fs)
    assert (n, b) == (LABEL_FILES, LABEL_BYTES), f"STOP: label con {n} file / {b} byte, attesi {(LABEL_FILES, LABEL_BYTES)}"
    tsha, _ = tree_sha256(dest)
    assert tsha == LABEL_TREE_SHA256, f"STOP: contenuto della label diverso da quello congelato (tree sha256 {tsha})"
    open(f"{WORK}/logs/label_count.txt", "w").write(f"seg={SEG} file={n} byte={b} tree_sha256={tsha} source={source_desc}\n")
    za = json.load(open(f"{dest}/{SEG}_inklabels.zarr/0/.zarray")); open(f"{WORK}/logs/label_zarray.json", "w").write(json.dumps(za))
    assert za["shape"] == LABEL_SHAPE, f"STOP: forma della label {za['shape']} diversa da {LABEL_SHAPE}"
    print(f"label verificata: file={n} byte={b} tree_sha256={tsha} ({source_desc}) shape={za['shape']}")

roots = glob.glob(f"/kaggle/input/**/{SEG}/{SEG}_inklabels.zarr/0/.zarray", recursive=True)
manifests = [m for m in glob.glob("/kaggle/input/**/manifest.json", recursive=True) if SEG in json.load(open(m)).get("segments", {})]
print("label montata:", roots[:1], "| manifest:", manifests[:1])
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
if roots and manifests:
    src = os.path.dirname(os.path.dirname(os.path.dirname(roots[0])))       # .../<SEG>
    man = json.load(open(manifests[0]))["segments"][SEG]
    files = man["files"]
    assert (len(files), sum(f["size"] for f in files)) == (LABEL_FILES, LABEL_BYTES), "STOP: manifest del dataset diverso dalle costanti congelate"
    missing = [f["path"] for f in files if not (os.path.isfile(os.path.join(src, f["path"][len(PREFIX):])) and os.path.getsize(os.path.join(src, f["path"][len(PREFIX):])) == f["size"])]
    assert not missing, f"STOP: {len(missing)} file della label mancanti o di dimensione diversa, p.es. {missing[:3]}"
    shutil.copytree(src, DEST)
    check_label_tree(DEST, f"kaggle_dataset manifest_sha256={hashlib.sha256(open(manifests[0], 'rb').read()).hexdigest()}")
else:
    assert KIND == "prep", "STOP: nei run GPU la label deve essere montata (nessun ripiego di rete con la GPU allocata)"
    def list_label_files():
        files, url = [], API
        while url:
            req = urllib.request.Request(url, headers={"User-Agent": "papyruslab-e02"})
            with urllib.request.urlopen(req, timeout=60) as r:
                files += [(e["path"], int(e["size"])) for e in json.load(r) if e.get("type") == "file"]
                m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", "") or "")
                url = m.group(1) if m else None
        return files
    def fetch(item):
        path, size = item
        assert path.startswith(PREFIX) and ".." not in path, f"STOP: percorso inatteso {path}"
        out = os.path.join(DEST, path[len(PREFIX):])
        if os.path.exists(out) and os.path.getsize(out) == size:
            return size
        os.makedirs(os.path.dirname(out), exist_ok=True)
        last = None
        for attempt in range(8):
            try:
                req = urllib.request.Request(RESOLVE + path, headers={"User-Agent": "papyruslab-e02"})
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                if len(data) == size:
                    open(out, "wb").write(data); return size
                last = f"dimensione {len(data)} != {size}"
            except Exception as ex:
                last = ex
            time.sleep(min(60, 5 * 2 ** attempt))
        raise RuntimeError(f"STOP: download fallito per {path}: {last}")
    files = list_label_files(); total = sum(s for _, s in files)
    assert (len(files), total) == (LABEL_FILES, LABEL_BYTES), f"STOP: label diversa dalla misura congelata: {len(files)} file, {total} byte"
    t0 = time.time()
    with cf.ThreadPoolExecutor(max_workers=4) as ex:
        got = sum(ex.map(fetch, files))
    print(f"scaricati {got} byte in {time.time() - t0:.0f} s")
    check_label_tree(DEST, "direct_download")


In [ ]:
%%bash
# Passo 5 (prep) — pooling ufficiale: livello 2 del volume 2,4 um, 84 piani centrali mediati a 4 a 4 -> 21 slice
set -e
source /kaggle/working/e02/env.sh
cd $HEAVY
disk_check "prima del pooling"
START=$(date +%s)
set -o pipefail
timeout -s INT -k 60 12600 python villa/ink-detection/scripts/prepare_9um_isotropic_input.py \
  "https://vesuvius-challenge-open-data.s3.amazonaws.com/PHerc1667/segments/20251212185248-w029_20251212185248662_flatboi/surface-volumes/2.399um-0.22m-78keV-volume-20251217075048.zarr" input/pherc1667-w029_pooled.zarr --level 2 --workers 4 2>&1 | tee $WORK/logs/prep_pherc1667-w029.log
EXIT=${PIPESTATUS[0]}; END=$(date +%s)
echo "exit_code=$EXIT durata_s=$((END-START))" | tee -a $WORK/logs/prep_pherc1667-w029.log
[ "$EXIT" -eq 0 ] || { echo "STOP: pooling terminato con exit_code=$EXIT"; exit 1; }
du -sh input/pherc1667-w029_pooled.zarr | tee -a $WORK/logs/prep_pherc1667-w029.log
disk_check "dopo il pooling"


In [ ]:
# Passo 5 (prep, segue) — verifica dell'input poolato contro la label, tar deterministico, hash
import json, os, io, tarfile, hashlib, time
import numpy as np, zarr
P = f"{HEAVY}/input/{SEG}_pooled.zarr"
g = zarr.open(P, mode="r"); a = g["0"]
lab = zarr.open(f"{HEAVY}/labels/{SEG}/{SEG}_inklabels.zarr", mode="r")["0"]
print("input", a.shape, a.dtype, a.chunks, "| label", lab.shape, "| attrs", dict(g.attrs))
assert tuple(a.shape) == tuple(lab.shape) == tuple(LABEL_SHAPE), f"STOP: input poolato {a.shape} vs label {lab.shape}"
assert list(g.attrs["source_z_slice"]) == [13, 97] and str(g.attrs["source_level"]) == "2" and str(a.dtype) == "uint8", g.attrs
assert g.attrs["source_shape_zyx"][0] == 109, "STOP: volume sorgente con profondita' diversa da 109"
cy, cx = a.shape[1] // 2, a.shape[2] // 2
blk = a[:, cy - 64: cy + 64, cx - 64: cx + 64]
assert blk.max() > 0, "STOP: blocco centrale vuoto"
zero_frac = float((a[10] == 0).mean())
print("frazione di zeri al piano 10:", round(zero_frac, 4))

def tree_sha256(root):
    per_file = {}
    for d, _, fs in os.walk(root):
        for f in fs:
            p = os.path.join(d, f)
            per_file[os.path.relpath(p, root).replace(os.sep, "/")] = hashlib.sha256(open(p, "rb").read()).hexdigest()
    h = hashlib.sha256()
    for rel, digest in sorted(per_file.items()):
        h.update(f"{rel}\n{digest}\n".encode())
    return h.hexdigest(), per_file

tsha, per_file = tree_sha256(P)
rels = sorted(per_file)
t0 = time.time()
tar_path = f"{WORK}/out/{SEG}_pooled.tar"
with tarfile.open(tar_path, mode="w", format=tarfile.PAX_FORMAT) as tf:      # deterministico: ordine, mtime 0, uid/gid 0
    for rel in rels:
        full = os.path.join(P, rel)
        info = tarfile.TarInfo(name=f"{SEG}_pooled.zarr/{rel}")
        info.size = os.path.getsize(full); info.mtime = 0; info.uid = info.gid = 0; info.uname = info.gname = ""; info.mode = 0o644
        with open(full, "rb") as fh:
            tf.addfile(info, fh)
tar_sha = hashlib.sha256(open(tar_path, "rb").read()).hexdigest()
tar_bytes = os.path.getsize(tar_path)
assert tar_bytes < 5 * 1024 ** 3, f"STOP: tar di {tar_bytes} byte oltre i 5 GB"
man = {"segment": SEG, "source_volume_url": SRC_URL, "source_level": "2", "attrs": dict(g.attrs), "shape": list(a.shape), "dtype": str(a.dtype),
       "chunks": list(a.chunks), "zero_fraction_plane10": zero_frac, "file_count": len(rels), "byte_total": sum(os.path.getsize(os.path.join(P, r)) for r in rels),
       "tree_sha256": tsha, "tar_name": os.path.basename(tar_path), "tar_bytes": tar_bytes, "tar_sha256": tar_sha,
       "tree_sha256_definition": "sha256 over sorted lines 'relpath\\nsha256(file)\\n', relpath relative to <SEG>_pooled.zarr/",
       "prepare_script_sha256": open(f"{WORK}/logs/prepare_script_sha256.txt").read().split()[0], "tar_seconds": round(time.time() - t0, 1)}
json.dump(man, open(f"{WORK}/out/prep_manifest_{SEG}.json", "w"), indent=1)
print(json.dumps({k: v for k, v in man.items() if k != "attrs"}, indent=1))


In [ ]:
%%bash
# Persistenza — hash di tutto cio' che viene conservato, stato finale
set -e
source /kaggle/working/e02/env.sh
cp /kaggle/working/e02/env.sh $WORK/logs/env.sh.txt
[ -f /kaggle/working/e02_guard.json ] && cp /kaggle/working/e02_guard.json $WORK/logs/guard.json
echo "end=$(date -u +%FT%TZ)" >> $WORK/logs/run_info.txt
disk_check "finale"
cd $WORK && find out logs -type f ! -name SHA256SUMS -print0 | sort -z | xargs -0 sha256sum > out/SHA256SUMS
cat out/SHA256SUMS
echo "persistito: $(du -sh $WORK | cut -f1)"
